In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "UseBSForNot/MergeInsthData_1211.csv"))
dim(data)

[1] 3227892      78

In [4]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "fac_pub"                                     
 [4] "year"                                        
 [5] "paper_type"                                  
 [6] "paper_language"                              
 [7] "nwbin"                                       
 [8] "new_word_reuse_times"                        
 [9] "npbin"                                       
[10] "new_phrase_reuse_times"                      
[11] "novelbin"                                    
[12] "rao_stirling"                                
[13] "author_id"                                   
[14] "author_position"                             
[15] "institution_id"                              
[16] "country_code"                                
[17] "country"                                     
[18] "num_author_log"                              
[19] "num_inst_log"                                
[20] "num_co

In [5]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                                     fac_pub 
                                           0 
                                        year 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                       nwbin 
                                           0 
                        new_word_reuse_times 
                                           0 
                                       npbin 
                                           0 
                      new_phrase_reuse_times 
                                           0 
                                    novelbin 
                                           0 
                                rao_stirling 
                                      303430 
                                   author_id 
                                           0 
                             author_position 
                                           0 
                              institution_id 
                                           0 
                                country_code 
                                           0 
                                     country 
                                           0 
                              num_author_log 
                                           0 
                                num_inst_log 
                                           0 
                             num_country_log 
                                           0 
                           num_reference_log 
                                           0 
                             timescited5_log 
                                      107147 
                            timescited10_log 
                                      163886 
                           timescitedall_log 
                                           0 
                               ab_length_log 
                                           0 
                     leadership_global_class 
                                           0 
                         mean_career_age_log 
                                           0 
                            inst_h_index_log 
                                           6 
                                   source_id 
                                           0 
                          source_h_index_log 
                                           0 
                                 source_type 
                                           0 
                                 core_source 
                                           0 
        Agricultural.and.Biological.Sciences 
                                           0 
                         Arts.and.Humanities 
                                           0 
Biochemistry..Genetics.and.Molecular.Biology 
                                           0 
         Business..Management.and.Accounting 
                                           0 
                        Chemical.Engineering 
                                           0 
                                   Chemistry 
                                           0 
                            Computer.Science 
                                           0 
                           Decision.Sciences 
                                           0 
                                   Dentistry 
                                           0 
                Earth.and.Planetary.Sciences 
                                           0 
         Economics..Econometrics.and.Finance 
                                           0 
                                      Energy 
                                         

In [6]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [7]:
# data <- data[!is.na(data$Atyp_10pct_Z), ]
# dim(data)
data <- data[data$num_reference >=5, ]
dim(data)

[1] 2768162      78

In [8]:
# data <- data[data$ab_length > 0, ]
# dim(data)
data <- data[data$year >= 1950, ]
dim(data)

[1] 2768162      78

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$decade <- (data$year %/% 10) * 10

In [11]:
data$leadership_global_class <- factor(data$leadership_global_class)
data <- within(data, leadership_global_class <- relevel(leadership_global_class, ref = 'shared'))
data$fac_pub <- factor(data$fac_pub)
data <- within(data, fac_pub <- relevel(fac_pub, ref = 'NonBSF'))
data$core_source <- factor(data$core_source)
data <- within(data, core_source <- relevel(core_source, ref = 'NonCore'))
data$decade <- as.factor(data$decade)

In [12]:
# variable type
paper_level <- "num_author_log + num_inst_log + num_country_log + num_reference_log + leadership_global_class + mean_career_age_log + inst_h_index_log" 
journal_level <- "source_h_index_log + core_source"
disciplines <- "Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [13]:
fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | year + author_id")
)

model_total <- feols(fml, data = data, vcov = "iid")
summary(model_total)

NOTE: 5 observations removed because of NA values (LHS: 1, RHS: 4).



OLS estimation, Dep. Var.: rao_stirling
Observations: 2,768,157
Fixed-effects: year: 76,  author_id: 78,875
Standard-errors: IID 
                                              Estimate Std. Error    t value
fac_pubBSF                                    0.002974   0.000080  37.064165
num_author_log                                0.000054   0.000061   0.884001
num_inst_log                                  0.001211   0.000078  15.545211
num_country_log                              -0.002973   0.000115 -25.961477
num_reference_log                             0.002141   0.000039  54.356877
leadership_global_classallNorth               0.001486   0.000117  12.668260
leadership_global_classallSouth               0.000807   0.000184   4.399553
mean_career_age_log                           0.000559   0.000065   8.616935
inst_h_index_log                              0.000351   0.000050   7.079320
source_h_index_log                           -0.000787   0.000029 -27.011958
core_sourceCore        

In [14]:
fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " + decade | author_id")
)

model_total <- feols(fml, data = data, vcov = "iid")
summary(model_total)

NOTE: 5 observations removed because of NA values (LHS: 1, RHS: 4).



OLS estimation, Dep. Var.: rao_stirling
Observations: 2,768,157
Fixed-effects: author_id: 78,875
Standard-errors: IID 
                                              Estimate Std. Error    t value
fac_pubBSF                                    0.002932   0.000080  36.464276
num_author_log                               -0.000495   0.000061  -8.151034
num_inst_log                                  0.000989   0.000078  12.673024
num_country_log                              -0.002775   0.000115 -24.177580
num_reference_log                             0.001580   0.000039  40.443124
leadership_global_classallNorth               0.001845   0.000118  15.695961
leadership_global_classallSouth               0.000616   0.000184   3.346850
mean_career_age_log                          -0.000423   0.000064  -6.595440
inst_h_index_log                              0.000537   0.000050  10.834885
source_h_index_log                           -0.000558   0.000029 -19.172310
core_sourceCore                   

In [17]:
MEs = ggemmeans(model_total, terms=c('fac_pub', 'decade'), typical='median')

In [18]:
MEs

x,predicted,std.error,conf.low,conf.high,group
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
NonBSF,0.1346616,0.0004168207,0.1338447,0.1354786,1950
NonBSF,0.1306008,0.0010807124,0.1284827,0.1327190,1960
NonBSF,0.1293962,0.0010383814,0.1273610,0.1314314,1970
NonBSF,0.1329245,0.0010268052,0.1309120,0.1349370,1980
NonBSF,0.1325031,0.0010219943,0.1305000,0.1345062,1990
NonBSF,0.1319104,0.0010193889,0.1299124,0.1339083,2000
NonBSF,0.1247673,0.0010181622,0.1227717,0.1267628,2010
NonBSF,0.1157575,0.0010186425,0.1137610,0.1177540,2020
BSF,0.1375941,0.0004171676,0.1367764,0.1384117,1950


In [19]:
fname = paste0(main_path, "UseBSForNot/predict_rao_1211_decade.csv")
write.csv(MEs, fname, row.names = FALSE)